# EDA — Liquidity Stress Prediction

Exploring the training data to understand structure, target balance, and which raw signals separate customers who go on to experience liquidity stress from those who don't.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 200)

train = pd.read_csv('../data/Train.csv')
test = pd.read_csv('../data/Test.csv')
data_dict = pd.read_csv('../data/data_dictionary.csv')

print('train shape:', train.shape)
print('test shape:', test.shape)

## Data dictionary structure

The 168 "behaviour" columns are 7 transaction types x 4 metrics x 6 months. Grouping them makes the schema much easier to reason about than scrolling 184 raw columns.

In [ ]:
data_dict.groupby('feature_group')['column_name'].apply(list).apply(len)

## Target distribution

~15% positive rate — imbalanced but not extreme.

In [ ]:
TARGET = 'liquidity_stress_next_30d'
print(train[TARGET].value_counts(normalize=True))
sns.countplot(x=TARGET, data=train)
plt.title('Liquidity Stress Distribution')
plt.show()

## Missingness and data types

No missing values in the training data at all — unusually clean.

In [ ]:
print(train.isnull().sum().sum(), 'total missing values')
print(train.dtypes.value_counts())

## Key signal: balance trend, not balance level

Comparing the most recent month's balance to 6 months ago separates the two classes far more clearly than any single month's raw balance.

In [ ]:
bal_cols = [f'm{i}_daily_avg_bal' for i in range(1, 7)]
bal_trend = train['m1_daily_avg_bal'] - train['m6_daily_avg_bal']
print('Balance trend (m1 - m6) by target:')
print(bal_trend.groupby(train[TARGET]).mean())

min_bal = train[bal_cols].min(axis=1)
print('\nRaw minimum balance across 6mo by target (much weaker signal):')
print(min_bal.groupby(train[TARGET]).mean())

## Key signal: withdraw-to-received ratio

Customers heading into stress withdraw a much larger share of what they receive.

In [ ]:
wd = train['m1_withdraw_total_value']
rc = train['m1_received_total_value'].replace(0, np.nan)
ratio = wd / rc
print('m1 withdraw/received ratio by target (median):')
print(ratio.groupby(train[TARGET]).median())

## Categorical features and IDs

Confirms one row per customer (no duplicate IDs) and the categorical value sets used for encoding.

In [ ]:
cat_cols = ['gender', 'region', 'smartphone', 'segment', 'earning_pattern']
for c in cat_cols:
    print(c, sorted(train[c].unique()))

print('\nDuplicate IDs in train:', train['ID'].duplicated().sum())

## Takeaways for feature engineering

- Build trend (slope, m1-m6 diff) and volatility (coefficient of variation) features per transaction type/metric, not just raw monthly values
- Cross-type ratios (withdraw/received, paybill/received, etc.) capture spending-vs-income behaviour
- No imputation strategy needed for missingness (there is none) — LightGBM handles the small number of NaNs produced by zero-division in ratio features natively
- Class imbalance (~15%) is moderate enough that explicit reweighting isn't needed — see `src/train.py` for a documented comparison

See `../src/features.py` and `../src/train.py` for the full implementation.